### MediLuzon Health Network — Readmission Analysis

**Note:** This analysis notebook was developed and executed on 
Databricks Community Edition after the Azure trial period ended. 
Table references use Unity Catalog three-part naming 
(`catalog.silver.fact_visits`, `catalog.gold.hospitals_kpi`) 
consistent with the production Azure environment.
This notebook reads from recreated Gold tables on Community Edition to produce the final analysis and key findings.


#### Network Analysis Summary

**Overall 30-day readmission rate**: 38.2%     
**Highest Specialization Admission Rate**: Oncology - 47.8%     
**Average length of stay**: 5.2 days    
**Average cost per visit**: PHP 57,548




In [0]:
from pyspark.sql.functions import (col, sum as spark_sum, avg, current_timestamp, round, round as spark_round, count
)
df_visits = spark.read.table("catalog.silver.fact_visits")

# Overall 30-day readmission rate
total = df_visits.count()
readmits = df_visits.filter(col("is_30day_readmission") == True).count()
overall_rate = readmits / total * 100
print(f"Overall 30-day readmission rate: {overall_rate:.1f}%")

Overall 30-day readmission rate: 38.2%


In [0]:
# Highest specialization admission rate
display(df_visits
    .filter(col("specialization").isNotNull())
    .groupBy("specialization")
    .agg(
        count("*").alias("total_visits"),
        spark_sum(col("is_30day_readmission").cast("int")).alias("readmissions"),
        round(spark_sum(col("is_30day_readmission").cast("int")) /
              count("*") * 100, 1).alias("readmission_rate_pct")
    )
    .orderBy("readmission_rate_pct", ascending=False))

specialization,total_visits,readmissions,readmission_rate_pct
Oncology,247,118,47.8
Endocrinology,303,133,43.9
Neurology,269,108,40.1
Infectious Disease,357,137,38.4
Pulmonology,270,102,37.8
Obstetrics,389,144,37.0
Cardiology,269,99,36.8
Nephrology,275,95,34.5
Gastroenterology,265,88,33.2
Internal Medicine,111,35,31.5


In [0]:
# Average length of stay
avg_los = df_visits.agg(round(avg("length_of_stay_days"), 1)).collect()[0][0]
print(f"Average length of stay: {avg_los} days")

Average length of stay: 5.2 days


In [0]:
# Average cost per visit
avg_cost = df_visits.agg(round(avg("total_cost_php"), 0)).collect()[0][0]
print(f"Average cost per visit: PHP {avg_cost:,.0f}")

Average cost per visit: PHP 57,548


#### Highest and lowest readmission by branch

The branch with the highest readmission rate is the **Davao branch** with a readmission rate of **41.5%**. The branch with the lowest readmission rate is the **Pampanga branch** with a readmission rate of **35.1%**.




In [0]:
# Readmission by branch
df_hospitals = spark.read.table("catalog.gold.hospitals_kpi")

display(df_hospitals
    .select("hospital_name", "total_visits", "total_readmissions", "readmission_rate")
    .orderBy("readmission_rate", ascending=False))

hospital_name,total_visits,total_readmissions,readmission_rate
MediLuzon General Hospital - Davao,270,112,0.415
MediLuzon General Hospital - Manila Flagship,321,127,0.396
MediLuzon Community Hospital - Cagayan de Oro,293,116,0.396
MediLuzon General Hospital - Quezon City,291,114,0.392
MediLuzon Specialist Center - Bonifacio Global City,294,115,0.391
MediLuzon Community Hospital - Iloilo,304,116,0.382
MediLuzon Community Hospital - Batangas,312,117,0.375
MediLuzon Specialist Center - Makati,282,104,0.369
MediLuzon General Hospital - Cebu,314,114,0.363
MediLuzon Community Hospital - Pampanga,319,112,0.351


#### Top 5 readmission by readmission volume and rate

The top 5 diagnosis in readmission by volume are:   
- **Essential Hypertension**    
- **Chronic Kidney Disease**    
- **Heart Failure**    
- **Type 2 Diabetes**   
- **Chemotherapy / Radiotherapy**   

The top 5 diagnosis in readmission by rate are:   
- **Chronic Kidney Disease (CKD)**    
- **Chemotherapy / Radiotherapy**     
- **Dialysis Care Encounter**     
- **Lung Cancer**    
- **Heart Failure**       


In [0]:
# Top 5 readmission by volume
df_diagnosis = spark.read.table("catalog.gold.diagnosis_kpi")

display(df_diagnosis
    .select("diagnosis_desc", "diagnosis_code", "total_cases", "total_readmissions", "readmission_rate")
    .orderBy("total_readmissions", ascending=False)
    .limit(5))

diagnosis_desc,diagnosis_code,total_cases,total_readmissions,readmission_rate
Essential Hypertension,I10,328,120,0.366
Chronic Kidney Disease (CKD),N18,175,119,0.68
Heart Failure,I50,197,96,0.487
Type 2 Diabetes Mellitus,E11,226,71,0.314
Chemotherapy / Radiotherapy,Z51,99,57,0.576


In [0]:
# Top 5 readmission by rate 
display(df_diagnosis
    .filter(col("total_cases") >= 10)
    .select("diagnosis_desc", "diagnosis_code", "total_cases", "total_readmissions", "readmission_rate")
    .orderBy("readmission_rate", ascending=False)
    .limit(5))

diagnosis_desc,diagnosis_code,total_cases,total_readmissions,readmission_rate
Chronic Kidney Disease (CKD),N18,175,119,0.68
Chemotherapy / Radiotherapy,Z51,99,57,0.576
Dialysis Care Encounter,Z49,99,53,0.535
Lung Cancer,C34,52,27,0.519
Heart Failure,I50,197,96,0.487


#### Highest specialization admission rate

**Oncology** specialization with a readmission rate of **47.8%** ranks as the highest specialization by readmission rate. **Endocrinology** ranks second with a readmission rate of **43.9%**.

In [0]:
# Specialization admission rate
df_physicians = spark.read.table("catalog.gold.physicians_kpi")

display(df_physicians
    .groupBy("specialization")
    .agg(
        spark_sum("total_visits").alias("total_visits"),
        spark_sum("total_readmissions").alias("total_readmissions"),
        round(spark_sum("total_readmissions") /
              spark_sum("total_visits") * 100, 1).alias("readmission_rate")
    )
    .orderBy("readmission_rate", ascending=False))

specialization,total_visits,total_readmissions,readmission_rate
Oncology,247,118,47.8
Endocrinology,303,133,43.9
Neurology,269,108,40.1
Infectious Disease,357,137,38.4
Pulmonology,270,102,37.8
Obstetrics,389,144,37.0
Cardiology,269,99,36.8
null,245,88,35.9
Nephrology,275,95,34.5
Gastroenterology,265,88,33.2
